# OpenCode Basic Session Example

Minimal, correct example of the OpenCode HTTP API: create a session,
send one message, print the reply, close the session.

**Source of truth for these request/response shapes**: `opencode`'s
own TypeScript source (`session/session.ts`, `session/prompt.ts`) plus
an empirical capture against a real running `opencode serve` instance
(`opencode-ai@1.18.3`) -- see `scripts/run_eval_client.py`'s module
docstring and `scripts/test_run_eval_client_e2e.py` in this repo for
the full verification trail. Prior draft notebooks in this folder
guessed at an `/api/session/...` path with a `DELETE` close and a
`{"model": "..."}` string body -- none of that matches the real API,
which has no `/api` prefix, closes sessions via `POST .../abort`, and
takes model as a `{"providerID", "modelID"}` object on every message,
not at session creation.

Endpoints actually used here:

- `POST {base_url}/session` -- create a session, body `{}`
- `POST {base_url}/session/{id}/message` -- send a turn
- `POST {base_url}/session/{id}/abort` -- end the session


In [1]:
import json
import os
import urllib.error
import urllib.request

OPENCODE_BASE_URL = os.environ.get("OPENCODE_BASE_URL", "http://localhost:4096")
OPENCODE_MODEL = os.environ.get("OPENCODE_MODEL", "local/ollama/qwen2.5-coder:7b")

if "/" not in OPENCODE_MODEL:
    raise ValueError(
        f"OPENCODE_MODEL={OPENCODE_MODEL!r} must be 'providerID/modelID' "
        "(e.g. 'local/ollama/qwen2.5-coder:7b' or 'anthropic/claude-sonnet-4-6')"
    )
PROVIDER_ID, MODEL_ID = OPENCODE_MODEL.split("/", 1)


def _request(base_url, method, path, body=None, timeout=60):
    trimmed = base_url.rstrip("/")
    url = f"{trimmed}{path}"
    data = json.dumps(body if body is not None else {}).encode("utf-8")
    req = urllib.request.Request(
        url, data=data, method=method,
        headers={"Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            raw = resp.read()
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", "replace")
        raise RuntimeError(f"{method} {path} -> HTTP {exc.code}: {detail}") from exc
    return json.loads(raw) if raw else {}


def create_session(base_url):
    resp = _request(base_url, "POST", "/session", {})
    session_id = resp.get("id") or resp.get("sessionID")
    if not session_id:
        raise RuntimeError(f"session creation response had no id/sessionID field: {resp}")
    return session_id


def send_message(base_url, session_id, provider_id, model_id, text, timeout=300):
    body = {
        "model": {"providerID": provider_id, "modelID": model_id},
        "parts": [{"type": "text", "text": text}],
    }
    return _request(base_url, "POST", f"/session/{session_id}/message", body, timeout=timeout)


def abort_session(base_url, session_id):
    return _request(base_url, "POST", f"/session/{session_id}/abort", {})


def extract_reply(response):
    """(assistant_text, tool_calls) from a SessionV1.WithParts response.
    Primary path (top-level "parts", filter on type == "text") confirmed
    empirically against a real opencode serve instance.
    """
    parts = response.get("parts")
    if parts is None and isinstance(response.get("message"), dict):
        parts = response["message"].get("parts")
    if parts is None:
        parts = []

    text_chunks = []
    tool_calls = []
    for part in parts:
        if not isinstance(part, dict):
            continue
        ptype = part.get("type", "")
        if ptype == "text":
            text_chunks.append(part.get("text", ""))
        elif "tool" in ptype.lower():
            tool_calls.append(part)
    return "\n".join(text_chunks), tool_calls


In [2]:
from IPython.display import Markdown, display

session_id = create_session(OPENCODE_BASE_URL)
display(Markdown(f"### Session created\n`{session_id}`"))

response = send_message(OPENCODE_BASE_URL, session_id, PROVIDER_ID, MODEL_ID, "Hello")
reply_text, tool_calls = extract_reply(response)

display(Markdown("## User\n\nHello"))
display(Markdown(f"## Assistant\n\n{reply_text}"))
if tool_calls:
    n = len(tool_calls)
    display(Markdown(f"_{n} tool call part(s) in response -- unverified branch, inspect `tool_calls` directly._"))

abort_session(OPENCODE_BASE_URL, session_id)
display(Markdown("### Session closed"))


### Session created
`ses_mock123`

## User

Hello

## Assistant

Hello from mock opencode.

### Session closed